# Notebook 09 — Stage 4 C1 Layer 1: Tirosh-subset cluster-formation cross-check

Stage 4 C1 Layer 1 (per `docs/stage4_dataset_schema.md` §5, refined).

## Scientific question
Stage 2 on GSE72056 (Tirosh, log2(TPM/10+1)) recovered only Melanocytic + Undifferentiated Tsoi states; no Neural crest-like or Transitory (Stage 3 P1).

Layer 1 tests whether the SAME Tirosh cells, processed from GSE115978's raw counts (Jerby-Arnon's re-deposit) through normalize_total + log1p + Stage 2-style Harmony/Leiden pipeline, recover NC/Transitory clusters.

- If GSE115978 Tirosh-subset ALSO fails to recover NC/Trans → confirms the limitation is cohort/patient composition, robust to source matrix and normalization (not a data-processing artifact).
- If GSE115978 Tirosh-subset DOES recover NC/Trans → processing/normalization matters for cluster formation even when NGFR distribution is similar.

## Prior context: normalization is NOT a confound (verified)
NGFR-positive rate in Tirosh malignant cells is consistent across two independent source matrices and normalization schemes:
- GSE72056 log2(TPM/10+1), n=1257: NGFR+ 10.0%, 9 patients, top patient 27.0%
- GSE115978 raw counts + normalize_total+log1p, n=1158: NGFR+ 8.5%, 9 patients, top patient 31.6%

Patient identity of top NGFR contributors matches (Mel79/89/81/53). This dual-source agreement strengthens Stage 3 P1's "NGFR distributed across patients" finding — robust to both source and normalization.

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
from pathlib import Path
import matplotlib.pyplot as plt
sc.settings.verbosity = 2

# Full Jerby-Arnon AnnData rebuild (same as notebook 08 Cell 4-6)
counts_path = Path('../data/raw/jerby_arnon/GSE115978_counts.csv.gz')
ann_path = Path('../data/raw/jerby_arnon/GSE115978_cell.annotations.csv.gz')

print('Loading GSE115978 counts + annotations...')
counts_df = pd.read_csv(counts_path, compression='gzip', index_col=0)
ann_df = pd.read_csv(ann_path, compression='gzip', index_col=0)
print(f'Counts: {counts_df.shape} (genes x cells)')
print(f'Annotations: {ann_df.shape}')

Loading GSE115978 counts + annotations...


Counts: (23686, 7186) (genes x cells)
Annotations: (7186, 6)


In [2]:
# Build full AnnData
adata_full = ad.AnnData(
    X=counts_df.T.values.astype(np.float32),
    obs=ann_df.loc[counts_df.columns],
    var=pd.DataFrame(index=counts_df.index),
)

# Layer 1 subset: Tirosh cohort x malignant
mask = (adata_full.obs['Cohort'] == 'Tirosh') & (adata_full.obs['cell.types'] == 'Mal')
adata_l1 = adata_full[mask].copy()
print(f'Layer 1 (Tirosh x Mal from GSE115978): {adata_l1.shape}')
print(f'Raw counts check, X.max(): {adata_l1.X.max()}')  # should be >> 20 (raw counts)
print()
print('Patient (samples) distribution:')
print(adata_l1.obs['samples'].value_counts())
print(f'Distinct patients: {adata_l1.obs["samples"].nunique()}')

Layer 1 (Tirosh x Mal from GSE115978): (1158, 23686)
Raw counts check, X.max(): 159721.0

Patient (samples) distribution:
samples
Mel79    486
Mel88    124
Mel78    120
Mel81    120
Mel89    100
Mel80     92
Mel71     62
Mel84     16
Mel53     14
Mel60     12
Mel94      6
Mel82      6
Name: count, dtype: int64
Distinct patients: 12


## Phase 1 — Cell ID intersection with Stage 2 annotated cells

Map GSE115978 Tirosh x Mal cells to Stage 2's GSE72056 annotated cells (`tirosh_malignant_annotated.h5ad`, n=1257) to assess overlap.

Note: cell ID naming conventions differ between GSE72056 (Tirosh original) and GSE115978 (Jerby-Arnon re-deposit). Check whether direct intersection is possible or whether ID translation is needed.

In [3]:
# Load Stage 2 annotated h5ad
stage2_path = Path('../data/processed/tirosh_malignant_annotated.h5ad')
adata_stage2 = sc.read_h5ad(stage2_path)
print(f'Stage 2 annotated: {adata_stage2.shape}')
print(f'Stage 2 obs columns: {adata_stage2.obs.columns.tolist()}')
print(f'Stage 2 tsoi_state distribution:')
print(adata_stage2.obs['tsoi_state'].value_counts())
print()

# Cell ID formats
print('Stage 2 cell IDs (first 5):')
print(adata_stage2.obs.index[:5].tolist())
print('Layer 1 (GSE115978) cell IDs (first 5):')
print(adata_l1.obs.index[:5].tolist())
print()

# Attempt direct intersection
stage2_ids = set(adata_stage2.obs.index)
l1_ids = set(adata_l1.obs.index)
direct_overlap = stage2_ids & l1_ids
print(f'Direct cell ID overlap: {len(direct_overlap)}')
print(f'(Stage 2: {len(stage2_ids)}, Layer 1: {len(l1_ids)})')

# If direct overlap is low, report ID format difference for translation analysis
if len(direct_overlap) < 100:
    print()
    print('WARNING: Low direct overlap. Cell ID formats likely differ.')
    print('Stage 2 sample ID:', list(stage2_ids)[:3])
    print('Layer 1 sample ID:', list(l1_ids)[:3])

Stage 2 annotated: (1257, 23686)
Stage 2 obs columns: ['tumor_id', 'malignant_status', 'cell_type_code', 'malignant_label', 'cell_type_label', 'patient', 'n_genes_by_counts', 'total_counts', 'leiden_r0.1', 'leiden_r0.2', 'leiden_r0.3', 'leiden_r0.5', 'leiden_r0.8', 'leiden_r1.0', 'tsoi_state']
Stage 2 tsoi_state distribution:
tsoi_state
Other (batch?)      704
Melanocytic         520
Undifferentiated     33
Name: count, dtype: int64

Stage 2 cell IDs (first 5):
['Cy71_CD45_D08_S524_comb', 'Cy81_FNA_CD45_B01_S301_comb', 'Cy80_II_CD45_B07_S883_comb', 'Cy81_Bulk_CD45_B10_S118_comb', 'Cy71_CD45_B05_S497_comb']
Layer 1 (GSE115978) cell IDs (first 5):
['cy78_CD45_neg_1_B04_S496_comb', 'cy79_p4_CD45_neg_PDL1_neg_E11_S1115_comb', 'CY88_5_B10_S694_comb', 'cy79_p1_CD45_neg_PDL1_pos_AS_C1_R1_F07_S67_comb', 'cy78_CD45_neg_3_H06_S762_comb']

Direct cell ID overlap: 366
(Stage 2: 1257, Layer 1: 1158)


## Phase 2 — Stage 2-style pipeline on GSE115978 Tirosh-subset

Design decision (mentor, 2026-05-27): cluster-level independent comparison.
Cell-level ID mapping between GSE72056 and GSE115978 is unreliable (~370/1158,
see Phase 1). So we run the Stage 2 pipeline independently on adata_l1 and
compare the resulting tsoi_state distribution to Stage 2's GSE72056 result at
the cohort level.

Convention locks (mirror Stage 2 / notebook 08 Layer 2):
- Normalize: normalize_total(1e6) + log1p (verified non-confound vs Tirosh TPM)
- HVG: flavor='seurat', 2000 genes
- Scale (max_value=10) -> PCA 50 comps
- Harmony: batch_key='samples', theta=2 (ascontiguousarray + direct Z_corr)
- Neighbors: n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony'
- Leiden sweep {0.1, 0.2, 0.3, 0.5, 0.8, 1.0}
- Marker stats computed on normalized (un-scaled) adata_l1 for all 4 markers
  (avoids notebook 08 Cell 24 AXL z-score comparability caveat)

Core comparison (Phase 3, after annotation): do Neural crest-like / Transitory
clusters EMERGE? Catch-all category name need not match Stage 2's
'Other (batch?)' exactly — the question is NC/Trans emergence, not catch-all
count parity.

In [4]:
# Normalize (same scheme as Layer 2 / Phase 2.8)
sc.pp.normalize_total(adata_l1, target_sum=1e6)
sc.pp.log1p(adata_l1)
print(f'After normalize_total(1e6) + log1p, X dtype: {adata_l1.X.dtype}')
print(f'X mean: {adata_l1.X.mean():.3f}, X max: {adata_l1.X.max():.3f}')

normalizing counts per cell


    finished (0:00:00)


After normalize_total(1e6) + log1p, X dtype: float32
X mean: 0.932, X max: 12.693


In [5]:
# HVG on log-normalized data
sc.pp.highly_variable_genes(adata_l1, flavor='seurat', n_top_genes=2000)
print(f'HVGs selected: {adata_l1.var.highly_variable.sum()}')

# Check whether Tsoi markers made it into HVG (notebook 08: only AXL did)
for g in ['MITF', 'SOX10', 'NGFR', 'AXL']:
    in_hvg = bool(adata_l1.var.loc[g, 'highly_variable']) if g in adata_l1.var.index else 'NOT IN VAR'
    print(f'  {g} in HVG: {in_hvg}')

# HVG subset for downstream embedding (full adata_l1 kept for marker stats)
adata_l1_hvg = adata_l1[:, adata_l1.var.highly_variable].copy()
print(f'HVG-subset shape: {adata_l1_hvg.shape}')

extracting highly variable genes


    finished (0:00:00)


HVGs selected: 2000
  MITF in HVG: False
  SOX10 in HVG: False
  NGFR in HVG: True
  AXL in HVG: True
HVG-subset shape: (1158, 2000)


In [6]:
sc.pp.scale(adata_l1_hvg, max_value=10)
sc.tl.pca(adata_l1_hvg, n_comps=50, random_state=0)
print(f'PCA done. variance_ratio top 10: {adata_l1_hvg.uns["pca"]["variance_ratio"][:10]}')

computing PCA


    with n_comps=50


    finished (0:00:00)


PCA done. variance_ratio top 10: [0.04982704 0.03678557 0.02584049 0.01852453 0.01481679 0.01298763
 0.01155366 0.00907853 0.00853045 0.0079188 ]


In [7]:
import harmonypy as hm
import logging
print(f'Harmony input X_pca: {adata_l1_hvg.obsm["X_pca"].shape}, n_batches: {adata_l1_hvg.obs["samples"].nunique()}')

# Working pattern (notebook 08 / Stage 2 notebook 03): harmonypy 0.2.0 returns
# Z_corr as (n_cells, n_pcs); do NOT transpose. ascontiguousarray for stride safety.
X_pca_in = np.ascontiguousarray(adata_l1_hvg.obsm['X_pca'])
logging.disable(logging.INFO)
try:
    hm_out = hm.run_harmony(X_pca_in, adata_l1_hvg.obs, 'samples',
                            theta=2, random_state=0, verbose=False)
finally:
    logging.disable(logging.NOTSET)
adata_l1_hvg.obsm['X_pca_harmony'] = np.ascontiguousarray(hm_out.Z_corr)
assert adata_l1_hvg.obsm['X_pca_harmony'].shape == adata_l1_hvg.obsm['X_pca'].shape
print(f'Harmony done. X_pca_harmony: {adata_l1_hvg.obsm["X_pca_harmony"].shape}')

Harmony input X_pca: (1158, 50), n_batches: 12


Harmony done. X_pca_harmony: (1158, 50)


In [8]:
sc.pp.neighbors(adata_l1_hvg, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony', random_state=0)
sc.tl.umap(adata_l1_hvg, random_state=0)
print('UMAP done')

computing neighbors


/Users/youyouwu/miniforge3/envs/melanoma-scrnaseq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    finished (0:00:04)


computing UMAP


    finished (0:00:01)


UMAP done


In [9]:
resolutions = [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
for res in resolutions:
    key = f'leiden_r{res}'
    sc.tl.leiden(adata_l1_hvg, resolution=res, key_added=key, random_state=0)
    print(f'{key}: {adata_l1_hvg.obs[key].nunique()} clusters')

# Copy leiden labels to full adata_l1 (cell order preserved by boolean var-mask)
assert (adata_l1_hvg.obs_names == adata_l1.obs_names).all(), 'cell order mismatch'
for res in resolutions:
    key = f'leiden_r{res}'
    adata_l1.obs[key] = adata_l1_hvg.obs[key].values

running Leiden clustering


    finished (0:00:00)


leiden_r0.1: 7 clusters
running Leiden clustering


    finished (0:00:00)


leiden_r0.2: 8 clusters
running Leiden clustering


...

    finished (0:00:00)


leiden_r0.8: 15 clusters
running Leiden clustering


    finished (0:00:00)


leiden_r1.0: 15 clusters


/var/folders/v7/sk64vbls6fg9sxpscqd5t8jw0000gn/T/ipykernel_42706/986761608.py:4: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_l1_hvg, resolution=res, key_added=key, random_state=0)


In [10]:
Path('../results/figures').mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc.pl.umap(adata_l1_hvg, color='samples', ax=axes[0], show=False, legend_loc='on data', legend_fontsize=6)
axes[0].set_title('Layer 1 UMAP by patient')
sc.pl.umap(adata_l1_hvg, color='leiden_r0.1', ax=axes[1], show=False, legend_loc='on data')
axes[1].set_title('Leiden r=0.1')
sc.pl.umap(adata_l1_hvg, color='leiden_r0.3', ax=axes[2], show=False, legend_loc='on data')
axes[2].set_title('Leiden r=0.3')
plt.tight_layout()
plt.savefig('../results/figures/09_layer1_umap_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved UMAP overview: results/figures/09_layer1_umap_overview.png')

Saved UMAP overview: results/figures/09_layer1_umap_overview.png


In [11]:
tsoi_markers = ['MITF', 'SOX10', 'NGFR', 'AXL']

# Dotplots on full adata_l1 (all 4 markers accessible regardless of HVG membership)
for res in [0.1, 0.2, 0.3, 0.5, 0.8]:
    key = f'leiden_r{res}'
    sc.pl.dotplot(adata_l1, tsoi_markers, groupby=key, standard_scale='var', show=False)
    plt.savefig(f'../results/figures/09_layer1_dotplot_{key}.png', dpi=150, bbox_inches='tight')
    plt.close()
print('Saved per-resolution dotplots')

Saved per-resolution dotplots


In [12]:
# Per-cluster Tsoi marker stats — ALL from normalized (un-scaled) adata_l1
# (consistent threshold semantics for all 4 markers; no z-score caveat)
def expr_vec(adata, gene):
    idx = adata.var.index.get_loc(gene)
    x = adata.X[:, idx]
    return (x.toarray().flatten() if hasattr(x, 'toarray') else np.asarray(x).flatten())

for res in [0.1, 0.2, 0.3, 0.5, 0.8]:
    key = f'leiden_r{res}'
    print(f'\n=== {key} ===')
    for gene in tsoi_markers:
        if gene not in adata_l1.var.index:
            print(f'  {gene}: NOT IN DATASET')
            continue
        e = expr_vec(adata_l1, gene)
        df = pd.DataFrame({'cluster': adata_l1.obs[key].values, 'expr': e})
        agg = df.groupby('cluster', observed=True)['expr'].agg([
            ('mean', 'mean'),
            ('pct_expr', lambda x: (x > 0).mean() * 100),
        ])
        print(f'  {gene}:')
        print(agg.to_string())


=== leiden_r0.1 ===
  MITF:
             mean   pct_expr
cluster                     
0        5.142706  90.804598
1        5.789006  91.166078
2        4.981967  85.534591
3        4.085084  67.000000
4        5.974291  94.623656
5        3.050838  59.550562
6        3.510552  76.744186
  SOX10:
             mean   pct_expr
cluster                     
0        3.809916  84.195402
1        4.075474  86.219081
2        3.631083  86.163522
3        3.356898  71.000000
4        3.768759  89.247312
5        3.161800  69.662921
6        3.195231  81.395349
  NGFR:
             mean   pct_expr
cluster                     
0        0.298089   6.321839
1        0.073810   1.766784
2        0.713271  17.610063
3        0.370984   6.000000
4        0.246152   5.376344
5        0.565101   8.988764
6        0.981420  27.906977
  AXL:
             mean   pct_expr
cluster                     
0        2.536057  65.229885
1        2.271803  82.332155
2        2.406464  79.874214
3        1.389161  

In [13]:
# NGFR-positive cell distribution across clusters (mirror notebook 08 Cell 25)
ngfr = expr_vec(adata_l1, 'NGFR')
ngfr_pos = ngfr > 0
print(f'NGFR-positive cells in Layer 1: {int(ngfr_pos.sum())} / {adata_l1.n_obs} = {ngfr_pos.mean()*100:.1f}%')
print(f'(Phase 2.8 reported 98/1158 = 8.5% for this same subset)')

for res in [0.1, 0.2, 0.3, 0.5, 0.8]:
    key = f'leiden_r{res}'
    ca = adata_l1.obs[key].values
    pos_dist = pd.Series(ca[ngfr_pos]).value_counts()
    tot_dist = pd.Series(ca).value_counts()
    print(f'\n=== {key} — NGFR-positive distribution across clusters ===')
    for cl in tot_dist.index:
        n_p = int(pos_dist.get(cl, 0)); n_t = int(tot_dist[cl])
        print(f'  Cluster {cl}: {n_p}/{n_t} = {n_p/n_t*100:.1f}% NGFR+')

NGFR-positive cells in Layer 1: 98 / 1158 = 8.5%
(Phase 2.8 reported 98/1158 = 8.5% for this same subset)

=== leiden_r0.1 — NGFR-positive distribution across clusters ===
  Cluster 0: 22/348 = 6.3% NGFR+
  Cluster 1: 5/283 = 1.8% NGFR+
  Cluster 2: 28/159 = 17.6% NGFR+
  Cluster 3: 6/100 = 6.0% NGFR+
  Cluster 4: 5/93 = 5.4% NGFR+
  Cluster 5: 8/89 = 9.0% NGFR+
  Cluster 6: 24/86 = 27.9% NGFR+

=== leiden_r0.2 — NGFR-positive distribution across clusters ===
  Cluster 0: 22/348 = 6.3% NGFR+
  Cluster 1: 5/283 = 1.8% NGFR+
  Cluster 2: 23/134 = 17.2% NGFR+
  Cluster 3: 6/100 = 6.0% NGFR+
  Cluster 4: 5/93 = 5.4% NGFR+
  Cluster 5: 8/89 = 9.0% NGFR+
  Cluster 6: 24/86 = 27.9% NGFR+
  Cluster 7: 5/25 = 20.0% NGFR+

=== leiden_r0.3 — NGFR-positive distribution across clusters ===
  Cluster 0: 22/348 = 6.3% NGFR+
  Cluster 1: 5/283 = 1.8% NGFR+
  Cluster 2: 15/118 = 12.7% NGFR+
  Cluster 3: 6/100 = 6.0% NGFR+
  Cluster 4: 5/93 = 5.4% NGFR+
  Cluster 5: 24/86 = 27.9% NGFR+
  Cluster 6: 14/6

## Phase 3 — Cluster 6 patient check + Tsoi state annotation (working res r=0.1)

Layer 1 finding (Phase 2): GSE115978 Tirosh-subset recovers no NC/Transitory clusters. Max-NGFR cluster (cluster 6, 27.9% NGFR) retains MITF 77% + SOX10 81% — Melanocytic identity, not dedifferentiated.

Phase 3: (a) verify cluster 6 NGFR is not single-patient-driven (parallel to Layer 2 Mel110 check); (b) assign Tsoi states at r=0.1.

In [14]:
# Cluster 6 = max NGFR cluster in Layer 1
key = 'leiden_r0.1'
cl6_mask = adata_l1.obs[key] == '6'
print(f'Cluster 6 size: {cl6_mask.sum()}')
print()
print('Cluster 6 patient composition:')
cl6_patients = adata_l1.obs.loc[cl6_mask, 'samples'].value_counts()
print(cl6_patients)
print(f'\nDistinct patients in cluster 6: {len(cl6_patients)}')
print(f'Top patient fraction: {cl6_patients.iloc[0] / cl6_mask.sum() * 100:.1f}%')
print()

# NGFR+ cells within cluster 6, per patient
gene_idx = adata_l1.var.index.get_loc('NGFR')
ngfr_expr = adata_l1.X[:, gene_idx]
if hasattr(ngfr_expr, 'toarray'):
    ngfr_expr = ngfr_expr.toarray().flatten()
else:
    ngfr_expr = np.array(ngfr_expr).flatten()
ngfr_pos_mask = ngfr_expr > 0

cl6_ngfr_mask = cl6_mask.values & ngfr_pos_mask
print(f'NGFR+ cells in cluster 6: {cl6_ngfr_mask.sum()}')
print('NGFR+ in cluster 6, per patient:')
cl6_ngfr_patients = adata_l1.obs.loc[cl6_ngfr_mask, 'samples'].value_counts()
print(cl6_ngfr_patients)
print()

# Compare to Layer 2 Mel110: was 91% single-patient
print('PARALLEL CHECK vs Layer 2:')
print('  Layer 2 NGFR+ : 91% from single patient Mel110')
top_frac = cl6_ngfr_patients.iloc[0] / cl6_ngfr_mask.sum() * 100 if cl6_ngfr_mask.sum() > 0 else float('nan')
print(f'  Layer 1 cluster 6 NGFR+: top patient fraction = {top_frac:.1f}%')

Cluster 6 size: 86

Cluster 6 patient composition:
samples
Mel81    81
Mel79     2
Mel82     2
Mel53     1
Mel60     0
Mel71     0
Mel78     0
Mel80     0
Mel84     0
Mel88     0
Mel89     0
Mel94     0
Name: count, dtype: int64

Distinct patients in cluster 6: 12
Top patient fraction: 94.2%

NGFR+ cells in cluster 6: 24
NGFR+ in cluster 6, per patient:
samples
Mel81    20
Mel82     2
Mel53     1
Mel79     1
Mel60     0
Mel71     0
Mel78     0
Mel80     0
Mel84     0
Mel88     0
Mel89     0
Mel94     0
Name: count, dtype: int64

PARALLEL CHECK vs Layer 2:
  Layer 2 NGFR+ : 91% from single patient Mel110
  Layer 1 cluster 6 NGFR+: top patient fraction = 83.3%


In [15]:
# Layer 1 overall NGFR+ per patient (consistency check vs Phase 2.8)
print('Layer 1 (all 1158 cells) NGFR+ per patient:')
l1_ngfr_patients = adata_l1.obs.loc[ngfr_pos_mask, 'samples'].value_counts()
print(l1_ngfr_patients)
print(f'\nDistinct patients with NGFR+: {len(l1_ngfr_patients)}')
print(f'Top patient fraction: {l1_ngfr_patients.iloc[0] / ngfr_pos_mask.sum() * 100:.1f}%')
print()
print('Comparison:')
print('  Layer 1 (Tirosh-subset): NGFR+ distributed, top patient ~32% (expect)')
print('  Layer 2 (New cohort): NGFR+ 91% Mel110 (single tumor)')
print('  -> Tirosh lacks a single NGFR-rich tumor like Mel110')

Layer 1 (all 1158 cells) NGFR+ per patient:
samples
Mel79    31
Mel89    26
Mel81    20
Mel53    14
Mel80     2
Mel82     2
Mel78     1
Mel88     1
Mel94     1
Mel60     0
Mel71     0
Mel84     0
Name: count, dtype: int64

Distinct patients with NGFR+: 12
Top patient fraction: 31.6%

Comparison:
  Layer 1 (Tirosh-subset): NGFR+ distributed, top patient ~32% (expect)
  Layer 2 (New cohort): NGFR+ 91% Mel110 (single tumor)
  -> Tirosh lacks a single NGFR-rich tumor like Mel110


## Tsoi state annotation (working resolution r=0.1, 7 clusters)

| Cluster | n | MITF% | SOX10% | NGFR% | AXL pattern | Tsoi state |
|---|---|---|---|---|---|---|
| 0 | 348 | 91% | 84% | 6.3% | 65% | Melanocytic |
| 1 | 283 | 91% | 86% | 1.8% | 82% | Melanocytic |
| 2 | 159 | 86% | 86% | 17.6% | 80% | Melanocytic |
| 3 | 100 | 67% | 71% | 6.0% | 46% | Melanocytic |
| 4 | 93 | 95% | 89% | 5.4% | 76% | Melanocytic |
| 5 | 89 | 60% | 70% | 9.0% | 81% (AXL high) | Undifferentiated |
| 6 | 86 | 77% | 81% | 27.9% | 77% | Melanocytic (NGFR tail) |

Annotation rationale:
- Clusters 0-4: Melanocytic (MITF/SOX10 high)
- Cluster 5: Undifferentiated (AXL-high, MITF/SOX10 lower) — corresponds to Stage 2's Undiff (33 cells)
- Cluster 6: Melanocytic despite 27.9% NGFR — MITF 77% + SOX10 81% retain melanocytic identity; not Neural crest-like (true NC requires MITF/SOX10 collapse, cf. Layer 2 NC: NGFR 61% + MITF 26%)
- **No Transitory, no Neural crest-like recovered** — consistent with Stage 2 GSE72056

In [16]:
tsoi_state_map = {
    '0': 'Melanocytic',
    '1': 'Melanocytic',
    '2': 'Melanocytic',
    '3': 'Melanocytic',
    '4': 'Melanocytic',
    '5': 'Undifferentiated',
    '6': 'Melanocytic',
}

adata_l1.obs['tsoi_state'] = (
    adata_l1.obs['leiden_r0.1'].astype(str).map(tsoi_state_map).astype('category')
)
adata_l1_hvg.obs['tsoi_state'] = adata_l1.obs['tsoi_state'].values

print('Layer 1 Tsoi state distribution:')
print(adata_l1.obs['tsoi_state'].value_counts())
print()
print('Comparison with Stage 2 GSE72056:')
print('  Stage 2: Melanocytic 520, Undifferentiated 33, Other(batch?) 704')
print('  Layer 1: see above')
print('  Both: NO Neural crest-like, NO Transitory')

Layer 1 Tsoi state distribution:
tsoi_state
Melanocytic         1069
Undifferentiated      89
Name: count, dtype: int64

Comparison with Stage 2 GSE72056:
  Stage 2: Melanocytic 520, Undifferentiated 33, Other(batch?) 704
  Layer 1: see above
  Both: NO Neural crest-like, NO Transitory


In [17]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc.pl.umap(adata_l1_hvg, color='leiden_r0.1', ax=axes[0], show=False, legend_loc='on data')
axes[0].set_title('Layer 1 Leiden r=0.1')
sc.pl.umap(adata_l1_hvg, color='tsoi_state', ax=axes[1], show=False, legend_loc='right margin')
axes[1].set_title('Layer 1 Tsoi state (Tirosh-subset GSE115978)')
plt.tight_layout()
plt.savefig('../results/figures/09_layer1_tsoi_state.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved Tsoi state UMAP')

Saved Tsoi state UMAP


## Layer 1 — Summary

Layer 1 applies Stage 2-style pipeline (normalize_total + log1p, HVG, Harmony, Leiden) to GSE115978's Tirosh-cohort malignant cells (n=1158, 12 patients), the same tumors Stage 2 analyzed from GSE72056.

### Finding: NC/Transitory do not emerge, robust to source + normalization

| | Stage 2 GSE72056 (TPM/log2) | Layer 1 GSE115978 (counts/log1p) | Layer 2 New cohort |
|---|---|---|---|
| Melanocytic | ✓ (520) | ✓ (1069) | ✓ (376) |
| Undifferentiated | ✓ (33) | ✓ (89, cluster 5) | (Ambiguous 42) |
| Neural crest-like | ✗ | ✗ | ✓ (72, Mel110) |
| Transitory | ✗ | ✗ | ✓ (28, Mel110) |
| Max-NGFR cluster | n/a | 27.9% (MITF 77%, not NC) | 61% (MITF 26%, true NC) |

The same Tirosh tumors fail to recover NC/Transitory under both GSE72056 (TPM, log2) and GSE115978 (raw counts, log1p) processing. Normalization scheme and source matrix are therefore ruled out as the cause.

### Mechanism: cohort/patient composition

Layer 1 NGFR+ cells (98/1158, 8.5%) are distributed across 9 patients (top ~32%), with no single NGFR-rich tumor. Cluster 6 (max NGFR 27.9%) retains Melanocytic identity (MITF 77%, SOX10 81%). By contrast, Layer 2's NC/Transitory recovery is 91% driven by a single tumor (Mel110). The Tirosh cohort simply lacks a Mel110-equivalent NGFR-rich dedifferentiated tumor; Mel53 (100% NGFR+ but only 14 cells) is too small to form a cluster.

### Mechanism refinement: lack of dedifferentiated tumors, not lack of NGFR-rich tumors

Cluster 6 (max-NGFR cluster, 27.9%) is 94.2% from a single patient (Mel81) — a single-patient NGFR-elevated cluster directly parallel to Layer 2's Mel110. But the two diverge critically:

| | Layer 2 Mel110 | Layer 1 Mel81 |
|---|---|---|
| Single-patient NGFR cluster | ✓ (91%) | ✓ (94%) |
| NGFR level | 61% | 27.9% |
| MITF / SOX10 | 26% / 2.8% (collapsed) | 77% / 81% (retained) |
| Dedifferentiated? | Yes → true NC/Transitory | No → Melanocytic |

This refines the mechanism. The Tirosh cohort is not simply missing an NGFR-rich tumor — it contains Mel81, an NGFR-elevated single-patient cluster. What Tirosh lacks is a tumor in which NGFR elevation coincides with MITF/SOX10 collapse. NGFR elevation alone (Mel81) does not produce Neural crest-like or Transitory states; dedifferentiation (Mel110) does. The limitation is the absence of dedifferentiated tumors, not the absence of NGFR signal.

Caveat: Cluster 6's 94% single-patient (Mel81) composition suggests it may carry patient-specific residual batch signal (parallel to Stage 2's "Other (batch?)" catch-all). However, whether cluster 6 is genuine Mel81 biology or residual batch, the conclusion is unchanged: it shows no dedifferentiation signature (MITF/SOX10 retained), so it does not constitute recovered NC/Transitory.

### Normalization is not a confound (dual-source verification)

NGFR+ rate is consistent across independent source matrices and normalization schemes:
- GSE72056 log2(TPM/10+1), n=1257: NGFR+ 10.0%, 9 patients, top 27.0%
- GSE115978 raw counts + normalize_total+log1p, n=1158: NGFR+ 8.5%, 9 patients, top 31.6%

Top NGFR-contributing patient identities match (Mel79/89/81/53). This strengthens Stage 3 P1's "NGFR distributed across patients" finding.

### Methodological note: cell-level mapping not viable

GSE72056 and GSE115978 use incompatible cell-barcode naming schemes (GSE72056: well + biopsy-type tokens; GSE115978: FACS + plate tokens; inconsistent casing; non-unique S-numbers). Direct cell-ID overlap reaches only ~370/1158 (32%). Layer 1 therefore uses cluster-level cohort comparison rather than cell-level mapping; patient-level correspondence (Mel79 ↔ patient 79) is exact and reliable.

In [18]:
adata_l1_hvg.write('../data/processed/layer1_tirosh_subset_annotated.h5ad')
adata_l1.write('../data/processed/layer1_tirosh_subset_full_annotated.h5ad')
print('Saved Layer 1 annotated h5ad (HVG + full)')
print('\n=== Phase 3 done ===')

Saved Layer 1 annotated h5ad (HVG + full)

=== Phase 3 done ===
